In [1]:
from transformer_lens import HookedTransformer

In [2]:
from examples import create_args, find_neurons, show_single_text

In [13]:
from neuron_analysis import neuron_analysis

In [3]:
DEVICE='cuda:0'

In [4]:
args = create_args(
    neuron_subset_name="weakening",
    intervention_type="zero_ablation",
    metric="entropy",
    topk=1,
    device=DEVICE,
    gate='-',
    post='+',
    #data_dir="../RW_functionalities_results",
)

In [5]:
model = HookedTransformer.from_pretrained(
    'allenai/OLMo-7B-0424-hf',
    #refactor_glu=True,
    device=DEVICE
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/226 [00:00<?, ?it/s]

Loaded pretrained model allenai/OLMo-7B-0424-hf into HookedTransformer


In [6]:
neuron_list = find_neurons(args)

In [7]:
input_ids = model.to_tokens(
    "Yesterday (21 December) the Government announced a package of support for hospitality and leisure businesses that are losing trade because of the Omic"
)#.flatten()

In [8]:
result_str, cache_clean, cache_ablated = show_single_text(
    args=args,
    model=model,
    input_ids=input_ids,
    pos=input_ids.shape[1]-2,#ignore the last token, which is the ground-truth output
    neuron_list=neuron_list,
    return_cache=True,
)

running clean model...
running ablated model...
inspect logit diff...


In [9]:
print(result_str)

Input tokens:
["<|endoftext|>", "Yesterday", " (", "21", " December", ")", " the", " Government", " announced", " a", " package", " of", " support", " for", " hospitality", " and", " leisure", " businesses", " that", " are", " losing", " trade", " because", " of", " the", " O", "mic"]
Ground-truth output token:
mic

The clean model outputs:
mic
with score 21.135128021240234

The ablated model outputs:


with score 9.471413612365723
top tokens promoted by the neurons (overall effect):
["mic", "MIC", "Mic", "MI", "micro", "Micro", " Mic", "PEC", "NS", "mi", " mic", "yster", "ECD", "mn", "VO", "microm"]
top tokens suppressed by the neurons (overall effect):
[" the", " one", " old", " a", " older", " many", " individual", " make", " it", " them", " our", " their", " big", " others", " other", " early"]


In [23]:
(model.ln_final(cache_clean['blocks.31.hook_resid_post'][0,-1])@model.W_U.detach()[:,model.to_single_token('mic')]).item()

21.135135650634766

In [24]:
(model.ln_final(cache_ablated['blocks.31.hook_resid_post'][0,-1])@model.W_U.detach()[:,model.to_single_token('mic')]).item()

1.3977980613708496

score difference to explain, *without* layer norm:

In [25]:
(cache_clean['blocks.31.hook_resid_post'][0,-1]-cache_ablated['blocks.31.hook_resid_post'][0,-1])@model.W_U.detach()[:,model.to_single_token('mic')]

tensor(3.5791, device='cuda:0')

Note: I suspect llmtt computes the scores *with* layer norm, hence a mismatch.

## Neuron 31.2578
we already know from llmtt this neuron is relevant.

In [14]:
neuron_analysis(model, 31, 2578)

gate vs. linear similarity: 0.19408632814884186
gate vs. out similarity: -0.45624828338623047
lin vs. out similarity: -0.41817840933799744
most similar tokens for w_out:
        dot product
Kem        0.070418
Newman     0.055059
Dodge      0.053756
Salem      0.049722
Fors       0.049218
Pho        0.048548
Mang       0.046293
Ri         0.046200
Karn       0.045719
Prest      0.045437
most similar tokens for -w_out:
              dot product
 micro           0.624943
 Micro           0.579665
micro            0.534183
 mic             0.498626
 Mic             0.494360
Micro            0.489219
mic              0.443438
Mic              0.409216
 microm          0.390586
 microscopic     0.350804
most similar tokens for w_in:
             dot product
 micro          0.427889
 Micro          0.375135
micro           0.290706
Micro           0.288925
 mic            0.236249
 Mic            0.218860
 microp         0.194707
 microm         0.185191
Mic             0.175375
 microphone 

How is w_in related to 'mic'?

In [27]:
model.blocks[31].mlp.W_in.detach()[:,2578]@model.W_U.detach()[:,model.to_single_token('mic')]

tensor(0.1380, device='cuda:0')

when activated positively, gate and in both detect 'mic' and write minus 'mic' (next to other similar things)

In [11]:
for subkey in ['pre', 'pre_linear', 'post']:
    print(subkey)
    print('clean')
    print(cache_clean[f'blocks.31.mlp.hook_{subkey}'][0,-1,2578].item())#batch pos neuron
    print('ablated')
    print(cache_ablated[f'blocks.31.mlp.hook_{subkey}'][0,-1,2578].item())
    print('\n')

pre
clean
5.226319789886475
ablated
0.5835621356964111


pre_linear
clean
-1.5932294130325317
ablated
-0.038652703166007996


post
clean
-8.28222370147705
ablated
-0.014478557743132114




In the clean run, the gate detects 'mic' but 'in' does not, leading to a strengthening of 'mic'. In the ablated run, activations are in the same quadrant but much smaller.

In the clean run compared to the ablated run, the neuron increases the mic logit by approx. 8.28*0.443 = ...

In [ ]:
(
    cache_clean['blocks.31.mlp.hook_post'][0,-1,2578] - cache_ablated['blocks.31.mlp.hook_post'][0,-1,2578]
).item() * (
    model.blocks[31].mlp.W_out[2578,:].detach()@(model.W_U.detach()[:,model.to_single_token('mic')])
).item()

3.6662347257527017

This completely explains the score difference!

What about the residual stream before the last MLP?

In [ ]:
(
    (
    cache_clean['blocks.31.hook_resid_mid'][0,-1] - cache_ablated['blocks.31.hook_resid_mid'][0,-1]
    ) @ (
        model.W_U.detach()[:,model.to_single_token('mic')]
    )
).item()

0.7509430050849915

So it's really just the neuron.

Next big question: What causes this neuron to activate in one run and not the other?

## Going further back

In [35]:
from transformers.activations import ACT2FN

In [40]:
model.blocks[31].mlp.W_gate.shape

torch.Size([4096, 11008])

In [41]:
# def hypothetical_neuron_activation(residual):
#         #normalised_residual = residual
#         normalised_residual = model.blocks[31].ln2(residual)
#         return (ACT2FN[model.cfg.act_fn](normalised_residual @ model.blocks[31].mlp.W_gate[:,2578]) * (normalised_residual @ model.blocks[31].mlp.W_in[:,2578])).item()
def hypothetical_neuron_activation(residual):
    x = model.blocks[31].ln2(residual)
    pre_act = x @ model.blocks[31].mlp.W_gate[:,2578]
    pre_linear = x @ model.blocks[31].mlp.W_in[:,2578]
    return (model.blocks[31].mlp.act_fn(pre_act) * pre_linear + model.blocks[31].mlp.b_in[2578]).item()

In [42]:
print("neuron activations computed from earlier residual streams:")
for layer in reversed(range(32)):
    for loc in ["mid", "pre"]:
        print(f"layer {layer}, {loc}:")
        hypothetical_clean_activation = hypothetical_neuron_activation(cache_clean[f'blocks.{layer}.hook_resid_{loc}'][0,-1])
        hypothetical_ablated_activation = hypothetical_neuron_activation(cache_ablated[f'blocks.{layer}.hook_resid_{loc}'][0,-1])
        print("clean:", hypothetical_clean_activation)
        print("ablated:", hypothetical_ablated_activation)
        print("diff:", hypothetical_clean_activation - hypothetical_ablated_activation)

neuron activations computed from earlier residual streams:
layer 31, mid:
clean: -35.22541427612305
ablated: -0.06566575914621353
diff: -35.15974851697683
layer 31, pre:
clean: -31.285078048706055
ablated: -7.961403846740723
diff: -23.323674201965332
layer 30, mid:
clean: -16.5766658782959
ablated: -16.132625579833984
diff: -0.44404029846191406
layer 30, pre:
clean: -19.14860725402832
ablated: -7.767256259918213
diff: -11.381350994110107
layer 29, mid:
clean: -10.367904663085938
ablated: -8.395546913146973
diff: -1.9723577499389648
layer 29, pre:
clean: -12.377445220947266
ablated: -9.353221893310547
diff: -3.0242233276367188
layer 28, mid:
clean: -7.535668849945068
ablated: -6.3218841552734375
diff: -1.2137846946716309
layer 28, pre:
clean: -7.948452472686768
ablated: -6.886883735656738
diff: -1.0615687370300293
layer 27, mid:
clean: -5.627388000488281
ablated: -4.5537919998168945
diff: -1.0735960006713867
layer 27, pre:
clean: -5.919015407562256
ablated: -5.716259956359863
diff: -0.2

something's wrong here